# Single-cell immunometabolic profiling of human T cells

Protein synthesis-based metabolic profiling coupled with high-dimensional spectral flow cytometry.

## Principle

Protein synthesis is measured by puromycin incorporation following a 30-min pulse. Acute metabolic inhibition during puromycin incorporation is used to quantify metabolic pathway dependencies.

* **Protein synthesis:** `Puro(DMSO) − Puro(CHX)`
* **Mitochondrial dependence:** `[(Puro(DMSO) − Puro(Oligomycin)) / (Puro(DMSO) − Puro(CHX)] × 100`

The spectral flow panel combines puromycin detection with phenotypic markers, enabling metabolic profiling of T-cell populations identified by PARC clustering.


## Example dataset

Healthy-donor PBMCs analyzed under DMSO, oligomycin, cycloheximide, and no-puromycin control conditions. Live CD3+ single cells were exported for downstream analysis.


# Load libraries, paths, dataset,  dictionnary

In [ ]:
# Ensure compatible versions of pandas, and FlowCytometryTools installed
# Import libraries

!pip install pandas
!pip install FlowCytometryTools
from collections import namedtuple
!pip install parc
!pip install hnswlib
!pip install fcsparser
!pip install seaborn
!pip install umap-learn
import collections
from collections.abc import MutableMapping
collections.MutableMapping = MutableMapping
import FlowCytometryTools
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import fcsparser
import seaborn as sns
from parc import PARC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import MinMaxScaler
import hnswlib
import umap
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from collections import Counter
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.image as mpimg

In [ ]:
# ==========================================
# INPUT / OUTPUT DIRECTORIES
# ==========================================

FCS_DIR = "../data"
OUTPUT_DIR = "../outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"FCS files: {os.path.abspath(FCS_DIR)}")
print(f"Outputs will be saved to: {os.path.abspath(OUTPUT_DIR)}")


# ==========================================
# STEP 1. LOAD FCS FILES
# ==========================================

all_data = []

for filename in os.listdir(FCS_DIR):
    if filename.endswith(".fcs"):
        fcs_file_path = os.path.join(FCS_DIR, filename)

        try:
            meta, data = fcsparser.parse(fcs_file_path)
        except Exception as e:
            print(f"Error parsing {filename}: {e}")
            continue

        if len(data) == 0:
            print(f"No data found in {filename}. Skipping this file.")
            continue
        else:
            print(f"Data from {filename} (first 5 rows):")
            print(data.head())

        df = pd.DataFrame(data)

        try:
            donor_id = filename.split("_")[0]
            drug_condition = filename.split("_")[1].split(".")[0]
        except IndexError:
            print(f"Filename format issue with {filename}. Skipping this file.")
            continue

        df["donor_ID"] = donor_id
        df["drug_condition"] = drug_condition

        flow_columns = [col for col in df.columns if "AF-" not in col]
        df_flow = df[flow_columns]

        all_data.append(df_flow)


# ==========================================
# STEP 2. CONCATENATE DATA
# ==========================================

if all_data:
    df_all = pd.concat(all_data, ignore_index=True)
    df_all.dropna(axis=0, how="any", inplace=True)

    print("Concatenated DataFrame (first 5 rows):")
    print(df_all.head())
else:
    print("No valid data to concatenate.")


# Check for NaN and infinite values
print("Checking for NaN and infinite values in the flow columns:")

for col in df_all.select_dtypes(include=[np.number]).columns:
    print(
        f"{col}: NaN values = {df_all[col].isna().sum()}, "
        f"Infinite values = {np.isinf(df_all[col]).sum()}"
    )

In [ ]:
# Define marker dictionary
channel_to_marker = {
    'FJComp-APC-Cy7-A': 'CD39',
    'FJComp-APC-Fire 810-A': 'CD27',
    'FJComp-Alexa Fluor 488-A': 'Puromycin',
    'FJComp-Alexa Fluor 647-A': 'CAR',
    'FJComp-Alexa Fluor 700-A': 'CD4',
    'FJComp-BB700-A': 'CD95',
    'FJComp-BUV395-A': 'TBET',
    'FJComp-BUV496-A': 'CD62L',
    'FJComp-BUV563-A': 'CD25',
    'FJComp-BUV615-A': 'CCR7',
    'FJComp-BUV661-A': 'CXCR3',
    'FJComp-BUV737-A': 'CD28',
    'FJComp-BUV805-A': 'CD69',
    'FJComp-BV421-A': 'CCR4',
    'FJComp-BV570-A': 'CD45RO',
    'FJComp-BV605-A': 'PD1',
    'FJComp-BV650-A': 'CD45RA',
    'FJComp-BV711-A': 'ICOS',
    'FJComp-BV750-A': 'TIM3',
    'FJComp-BV785-A': 'CCR5',
    'FJComp-LIVE DEAD Blue-A': 'LiveDead',
    'FJComp-PE-A': 'TCF1',
    'FJComp-PE-Cy5-A': 'CD3',
    'FJComp-PE-Cy7-A': 'HELIOS',
    'FJComp-PE-Dazzle594-A': '41BB',
    'FJComp-PE-Fire 700-A': 'LAG3',
    'FJComp-Pacific Blue-A': 'CD127',
    'FJComp-RB780-A': 'CD71',
    'FJComp-Spark YG593-A': 'CD8'
}


# Arcsinh transformation

In [ ]:
# ==========================================
# PRISM-LIKE FONT / EXPORT STYLE ONLY
# ==========================================

mpl.rcParams["font.family"] = "Arial"

mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["axes.titlesize"] = 6
mpl.rcParams["xtick.labelsize"] = 6
mpl.rcParams["ytick.labelsize"] = 6
mpl.rcParams["legend.fontsize"] = 6

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

sns.set_style("white")

# ==========================================
# TRANSFORMATION FUNCTION
# ==========================================

def arcsinh_to_percentage(data, scale=5.0, dynamic_scale=True):
    data = np.asarray(data, dtype=float)

    if dynamic_scale:
        max_data = np.nanmax(data)
        scale = max_data / 150.0

        if scale == 0 or np.isnan(scale):
            scale = 5.0

    transformed_data = np.arcsinh(data / scale)

    min_val = np.nanmin(transformed_data)
    max_val = np.nanmax(transformed_data)

    if max_val == min_val:
        return np.zeros_like(transformed_data)

    return ((transformed_data - min_val) / (max_val - min_val)) * 100


# ==========================================
# APPLY TRANSFORMATION
# ==========================================

df_transformed = df_all.copy()

metadata_cols = [
    "donor_ID",
    "drug_condition",
    "Time"
]

flow_columns = [
    col for col in df_all.columns
    if col not in metadata_cols
]

for col in flow_columns:
    df_transformed[col] = arcsinh_to_percentage(
        df_all[col],
        scale=5.0,
        dynamic_scale=True
    )


# ==========================================
# SELECT MARKERS
# ==========================================

exclude_columns = [
    "FJComp-LIVE DEAD Blue-A",
    "FJComp-Alexa Fluor 647-A",
    "FSC-A",
    "FSC-H",
    "SSC-B-A",
    "SSC-B-H",
    "SSC-H"
]

plot_columns = [
    col for col in flow_columns
    if not any(exclude in col for exclude in exclude_columns)
]

desired_order = [
    "Puromycin",
    "CD3",
    "CD4",
    "CD8",
    "CD45RA",
    "CD45RO",
    "CD25",
    "CD71",
    "CD69",
    "CCR7",
    "CD62L",
    "CD27",
    "CD127",
    "TCF1",
    "CD95",
    "CD28",
    "41BB",
    "ICOS",
    "PD1",
    "TIM3",
    "LAG3",
    "CD39",
    "CXCR3",
    "CCR4",
    "CCR5",
    "TBET",
    "HELIOS"
]

column_to_marker = {
    col: channel_to_marker.get(col, col)
    for col in plot_columns
}

ordered_columns = []

for marker in desired_order:
    for col, real_marker in column_to_marker.items():
        if real_marker == marker:
            ordered_columns.append(col)

ordered_column_names = [
    channel_to_marker.get(col, col)
    for col in ordered_columns
]


# ==========================================
# VIOLIN PLOT
# ==========================================

plt.figure(figsize=(4, 2))

sns.violinplot(
    data=df_transformed[ordered_columns],
    linewidth=0.25
)

plt.xticks(
    ticks=range(len(ordered_columns)),
    labels=ordered_column_names,
    rotation=90,
    ha="center",
    fontsize=6
)

plt.tick_params(axis="x", pad=1)

plt.yticks(fontsize=6)

plt.title("", fontsize=6)
plt.ylabel("Relative expression (%)", fontsize=6)
plt.xlabel("")


# ==========================================
# SAVE PDF ONLY
# ==========================================

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "ViolinPlot_RelativeExpression_ordered.pdf"
    ),
    dpi=600,
    bbox_inches="tight",
    format="pdf"
)

plt.show()

# PARC clustering - phenotypic markers

In [ ]:
#PARC clustering

# Remove metadata columns (donor_ID, drug_condition) and Puro for clustering
df_for_clustering = df_transformed.drop(columns=['donor_ID', 'FJComp-Alexa Fluor 488-A', 'drug_condition', 'Time'], errors='ignore')
print(df_for_clustering.head())

# Run PARC clustering
parc_model = PARC(data=df_for_clustering.values, resolution_parameter=0.5)  # Set resolution_parameter to control the size of clusters
clusters = parc_model.run_PARC()
print(clusters)

## UMAP - cluster visualization

In [ ]:
clusters = parc_model.labels
n_clusters = 18

umap_model = umap.UMAP(
    n_neighbors=45,
    min_dist=0.5,
    n_epochs=200,
    n_components=2,
    random_state=42,
    n_jobs=1
)

umap_result = umap_model.fit_transform(df_for_clustering)

df_umap = pd.DataFrame(
    umap_result,
    columns=["UMAP1", "UMAP2"]
)

df_umap["Cluster"] = clusters

In [ ]:
palette = sns.color_palette("tab20", n_colors=n_clusters)

fig, ax = plt.subplots(figsize=(3, 2.5))

sc = ax.scatter(
    df_umap["UMAP1"],
    df_umap["UMAP2"],
    c=df_umap["Cluster"],
    cmap=mcolors.ListedColormap(palette),
    s=0.5,
    alpha=0.7,
    linewidths=0,
    rasterized=True   # key line: makes points lightweight in PDF
)

for cluster_id in range(n_clusters):
    cluster_data = df_umap[df_umap["Cluster"] == cluster_id]
    centroid = cluster_data[["UMAP1", "UMAP2"]].mean(axis=0)

    ax.text(
        centroid["UMAP1"],
        centroid["UMAP2"],
        f"{cluster_id}",
        color="black",
        fontsize=4,
        ha="center",
        va="center"
    )

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("")

plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "UMAP_PARC_clusters.pdf"),
    dpi=300,              # 300 usually enough; 600 makes heavier
    bbox_inches="tight",
    format="pdf"
)

plt.show()

In [ ]:
# ============================================================
# SUBSETS 
# ============================================================

annotation_groups = {

    "CD4 Naive/SCM": {
        1:  "CD45RA+ CCR7+ TCF1+",
        10: "CD45RA+ CCR7+ TCF1+",
        8:  "CD45RA+ CCR7+ TCF1+ CXCR3+",
        11: "CD45RA+ CCR7+ TCF1+ PD1+ CXCR3+",
        16: "CD45RA+ CCR7+ TCF1− CD62L−",
    },

    "CD4 CM": {
        0:  "CD45RO+ CCR7+ CXCR3+",
        14: "CD45RO+ CCR7+ CCR4+ CD25+",
    },

    "CD4 EM": {
        3:  "CD45RO+ CCR7− CXCR3+",
        12: "CD45RO+ CCR7− PD1+ CXCR3+",
    },

    "CD4 Treg": {
        13: "CD25+ CD127− HELIOS+",
    },

    "CD8 Naive/SCM": {
        9: "CD45RA+ CCR7+ TCF1+",
    },

    "CD8 CM": {
        6:  "CD45RO+ CD62L+ CXCR3+",
        15: "CD45RO+ CD62Llow CXCR3+",
    },

    "CD8 EM": {
        7: "CD45RO+ CCR7− CCR5+",
    },

    "CD8 EMRA": {
        2:  "CD45RA+ CCR7− TCF1− TBET+",
        4:  "CD45RA+ CCR7− CXCR3+ CCR5+ TBET+",
        17: "CD45RA+ CCR7− TBEThi CD127−",
    },

    "DN T cells": {
        5: "CD4− CD8−",
    },
}
# ============================================================
# LOCAL VARIABLES ONLY — DOES NOT OVERWRITE PARC clusters
# ============================================================

umap_cluster_ids = sorted(df_umap["Cluster"].unique())
umap_n_clusters = len(umap_cluster_ids)

umap_palette = sns.color_palette("tab20", n_colors=umap_n_clusters)

umap_cluster_to_color = {
    cluster_id: umap_palette[i]
    for i, cluster_id in enumerate(umap_cluster_ids)
}

umap_point_colors = df_umap["Cluster"].map(umap_cluster_to_color)

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=(4.5, 2.5))

ax.scatter(
    df_umap["UMAP1"],
    df_umap["UMAP2"],
    c=umap_point_colors,
    s=0.15,
    alpha=0.7,
    linewidths=0,
    rasterized=True
)

# ============================================================
# GROUP LABELS ON UMAP
# ============================================================

for group_name, group_cluster_dict in annotation_groups.items():

    group_cluster_ids = list(group_cluster_dict.keys())

    group_data = df_umap[
        df_umap["Cluster"].isin(group_cluster_ids)
    ]

    if len(group_data) == 0:
        continue

    centroid = group_data[["UMAP1", "UMAP2"]].mean()

    ax.text(
        centroid["UMAP1"],
        centroid["UMAP2"],
        group_name,
        fontsize=5,
        fontweight="bold",
        color="black",
        ha="center",
        va="center"
    )

# ============================================================
# LEGEND
# ============================================================

legend_handles = []
legend_labels = []

for group_name, group_cluster_dict in annotation_groups.items():

    legend_handles.append(Line2D([], [], linestyle="none"))
    legend_labels.append(group_name)

    for cluster_id, annotation in group_cluster_dict.items():

        if cluster_id not in umap_cluster_to_color:
            continue

        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="none",
                markerfacecolor=umap_cluster_to_color[cluster_id],
                markeredgecolor="none",
                markersize=4,
            )
        )

        legend_labels.append(
            f"Cluster {cluster_id}: {annotation}"
        )

# ============================================================
# OPTIONAL: UNANNOTATED CLUSTERS
# ============================================================

annotated_cluster_ids = {
    cluster_id
    for group_cluster_dict in annotation_groups.values()
    for cluster_id in group_cluster_dict
}

unannotated_cluster_ids = [
    cluster_id for cluster_id in umap_cluster_ids
    if cluster_id not in annotated_cluster_ids
]

if len(unannotated_cluster_ids) > 0:

    legend_handles.append(Line2D([], [], linestyle="none"))
    legend_labels.append("Other")

    for cluster_id in unannotated_cluster_ids:

        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="none",
                markerfacecolor=umap_cluster_to_color[cluster_id],
                markeredgecolor="none",
                markersize=4,
            )
        )

        legend_labels.append(
            f"Cluster {cluster_id}: unannotated"
        )

# ============================================================
# LEGEND FORMAT
# ============================================================

legend = ax.legend(
    legend_handles,
    legend_labels,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=4.5,
    handletextpad=0.4,
    labelspacing=0.35,
    borderaxespad=0
)

for text in legend.get_texts():
    if text.get_text() in annotation_groups or text.get_text() == "Other":
        text.set_fontweight("bold")
        text.set_fontsize(5)

# ============================================================
# AXES
# ============================================================

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("")

sns.despine()

# ============================================================
# SAVE
# ============================================================

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "UMAP_PARC_clusters_grouped.pdf"
    ),
    dpi=1200,
    bbox_inches="tight",
    format="pdf"
)

plt.show()

## CLUSTERMAP 

In [ ]:
# ==========================================
# CLUSTER MEAN EXPRESSION MATRIX
# ==========================================

cluster_matrix = (
    df_transformed[plot_columns]
    .assign(PARC_LABEL=clusters)
    .groupby("PARC_LABEL")
    .mean()
)

cluster_matrix.columns = [
    channel_to_marker.get(col, col)
    for col in cluster_matrix.columns
]

# Force-remove cluster-like columns after marker renaming
cluster_matrix = cluster_matrix[
    [
        c for c in cluster_matrix.columns
        if c.lower() not in ["cluster", "parc", "parc_label"]
    ]
]

cluster_matrix.index.name = None


# ==========================================
# CLUSTERMAP
# ==========================================

fig_w = 3
fig_h = 3

xtick_fontsize = 4
ytick_fontsize = 4

cbar_pos = (0.02, 0.82, 0.012, 0.08)

g = sns.clustermap(
    cluster_matrix,
    cmap="RdBu_r",
    figsize=(fig_w, fig_h),
    row_cluster=True,
    col_cluster=True,
    linewidths=0.01,
    linecolor="black",
    dendrogram_ratio=(0.10, 0.08),
    tree_kws={"linewidths": 0.25},
    cbar_pos=cbar_pos
)

g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_ylabel("")

plt.setp(
    g.ax_heatmap.get_xticklabels(),
    rotation=90,
    ha="center",
    fontsize=xtick_fontsize
)

plt.setp(
    g.ax_heatmap.get_yticklabels(),
    rotation=0,
    fontsize=ytick_fontsize
)

g.ax_heatmap.tick_params(
    axis="x",
    labelsize=xtick_fontsize,
    width=0.4,
    length=1.5,
    pad=1
)

g.ax_heatmap.tick_params(
    axis="y",
    labelsize=ytick_fontsize,
    width=0.4,
    length=1.5,
    pad=1
)

g.cax.tick_params(
    labelsize=4,
    width=0.4,
    length=1.5
)

g.cax.set_ylabel("")
g.cax.set_title(
    "%",
    fontsize=4,
    pad=2
)

g.fig.subplots_adjust(
    left=0.12,
    bottom=0.25
)

g.cax.set_position(cbar_pos)

# ==========================================
# SAVE PDF ONLY
# ==========================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

g.fig.savefig(
    os.path.join(
        OUTPUT_DIR,
        "ClusterMap_RelativeExpression_byCluster_noPuromycin_noCluster_3x3.pdf"
    ),
    dpi=600,
    bbox_inches="tight",
    format="pdf"
)

plt.show()

## Cluster representation per drug 

In [ ]:
# ==========================================
# QUALITY CONTROL: CLUSTER REPRESENTATION
# ==========================================

df_transformed["cluster"] = clusters

cluster_counts = (
    df_transformed
    .groupby(["drug_condition", "cluster"])
    .size()
    .reset_index(name="count")
)

total_counts_per_condition = (
    df_transformed
    .groupby("drug_condition")["cluster"]
    .count()
    .reset_index(name="total_count")
)

cluster_counts = cluster_counts.merge(
    total_counts_per_condition,
    on="drug_condition"
)

cluster_counts["percentage"] = (
    cluster_counts["count"] / cluster_counts["total_count"]
) * 100

pivot_df = (
    cluster_counts
    .pivot(index="drug_condition", columns="cluster", values="percentage")
    .fillna(0)
)

custom_order = ["DMSO", "OLIGO", "CHX", "NP"]
pivot_df = pivot_df.reindex(custom_order)

# ==========================================
# PLOT
# ==========================================

ax = pivot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(2.8, 2.2),
    colormap="viridis",
    width=0.8
)

plt.grid(False)

plt.title("")
plt.xlabel("")
plt.ylabel("Cluster frequency (%)", fontsize=6)

plt.xticks(
    rotation=45,
    ha="right",
    fontsize=6
)

plt.yticks(fontsize=6)

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=5,
    title_fontsize=6,
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# ==========================================
# SAVE PDF ONLY
# ==========================================

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "ClusterDistribution_perDrugCondition.pdf"
    ),
    dpi=600,
    bbox_inches="tight",
    format="pdf"
)

plt.show()

## INSPECT clusters: violin plots All_cell VS clusterX

In [ ]:
# Violin plots

# Function to downsample data
def downsample_data(df, fraction=0.1):
    return df.sample(frac=fraction, random_state=42)

# Check if 'Cluster' column exists, else assign clusters
if 'Cluster' not in df_transformed.columns:
    df_transformed['Cluster'] = clusters

# Remove unwanted columns
remove_columns = ['FSC-A', 'FSC-H', 'SSC-A', 'SSC-B-A', 'SSC-B-H', 'SSC-H', 'FJComp-LIVE DEAD Blue-A']
filtered_columns = [col for col in flow_columns if col not in remove_columns and col != 'Cluster']

# Downsample all cells data for plotting
all_cells_data = downsample_data(df_transformed[filtered_columns], fraction=0.1)

# Get unique clusters
unique_clusters = df_transformed['Cluster'].unique()

# Define the desired order of markers (channels)
desired_order = [
    "CD3", "CD4", "CD8", "CD45RA", "CD45RO", "CD25", "CD71", "CD69", "CCR7", 
    "CD62L", "CD27", "CD127", "TCF1", "CD95", "CD28", "41BB", "ICOS", "PD1", "TIM3", 
    "LAG3", "CD39", "CXCR3", "CCR4", "CCR5", "TBET", "HELIOS"
]

# Function to create violin plots with desired order of channels
def create_violin_plots(cluster_of_interest, filtered_columns, all_cells_data, figsize=(8, 4)):
    # Get the data for the cluster of interest and downsample
    cluster_data = downsample_data(df_transformed[df_transformed['Cluster'] == cluster_of_interest][filtered_columns], fraction=0.1)

    # Map the filtered columns to their corresponding marker names using channel_to_marker
    channel_to_marker_mapping = {col: channel_to_marker.get(col, col) for col in filtered_columns}

    # Reorder the filtered columns based on the desired_order
    ordered_filtered_columns = sorted(filtered_columns, key=lambda col: desired_order.index(channel_to_marker_mapping.get(col, col)) if channel_to_marker_mapping.get(col, col) in desired_order else float('inf'))

    # Create the figure for the violin plot
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    
    # Loop over the ordered columns and create the violin plots
    for i, channel in enumerate(ordered_filtered_columns):
        all_cells = all_cells_data[channel]
        cluster_cells = cluster_data[channel]
        
        # Combine the data for both all cells and cluster cells
        combined_data = pd.DataFrame({
            'Value': np.concatenate([all_cells, cluster_cells]),
            'Group': ['All Cells'] * len(all_cells) + ['Cluster'] * len(cluster_cells),
            'Channel': [channel] * (len(all_cells) + len(cluster_cells))
        })

        # Map the channel to its real marker name using channel_to_marker
        combined_data['Channel'] = combined_data['Channel'].map(channel_to_marker).fillna(combined_data['Channel'])

        # Create the violin plot with updated parameter
        sns.violinplot(
            x='Channel', y='Value', hue='Group', data=combined_data,
            inner=None, linewidth=1.2, split=True, density_norm='count', palette=['lightgray', 'red'],
            legend=False, ax=ax
        )
    
    # Set x-axis labels based on the sorted order of markers
    # Explicitly set x-ticks using the range of ordered_filtered_columns
    ax.set_xticks(np.arange(len(ordered_filtered_columns)))
    ax.set_xticklabels([channel_to_marker.get(col, col) for col in ordered_filtered_columns], rotation=90, fontsize=8)
    
    # Set labels and title
    ax.set_xlabel('Channels', fontsize=8)
    ax.set_ylabel('Expression Value', fontsize=8)
    ax.set_title(f"Violin Plot: All Cells vs. Cluster {cluster_of_interest}", fontsize=10)

    # Adjust layout for better spacing
    plt.tight_layout()
    plt.show()

# Create violin plots for each cluster
for cluster_of_interest in unique_clusters:
    create_violin_plots(cluster_of_interest, filtered_columns, all_cells_data, figsize=(8, 4))


# Calculate Protein synthesis and Mitochondrial dependence in each cluster/donor condition

In [ ]:
# Calculate metabolic parameters in each cluster from df_all

# Step 1: Assign the clusters back to the original metadata DataFrame `df_all`
df_all['Cluster'] = clusters

# Step 2: Filter the relevant columns: 'Cluster', 'donor_ID', 'drug_condition', and 'FJComp-Alexa Fluor 488-A'
df_combined_filtered = df_all[['Cluster', 'donor_ID', 'drug_condition', 'FJComp-Alexa Fluor 488-A']]

# Step 3: Calculate the Mean Fluorescence Intensity (MFI) for 'FJComp-Alexa Fluor 488-A'
# for each combination of 'donor_ID', 'drug_condition', and 'Cluster'
cluster_mfi = df_combined_filtered.groupby(['donor_ID', 'drug_condition', 'Cluster'])['FJComp-Alexa Fluor 488-A'].mean().reset_index()

# Step 4: Create a results list to store the calculated protein synthesis and mitochondrial dependence
results = []
unique_clusters = cluster_mfi['Cluster'].unique()
unique_donors = cluster_mfi['donor_ID'].unique()
drug_conditions = ['DMSO', 'CHX', 'OLIGO']

# Step 5: Loop over each cluster and donor condition
for cluster in unique_clusters:
    for donor in unique_donors:

        # Step 6: Extract MFI values for the three conditions: DMSO, CHX, and OLIGO
        mfi_dms0 = cluster_mfi[(cluster_mfi['Cluster'] == cluster) &
                               (cluster_mfi['donor_ID'] == donor) &
                               (cluster_mfi['drug_condition'] == 'DMSO')]['FJComp-Alexa Fluor 488-A'].mean()

        mfi_chx = cluster_mfi[(cluster_mfi['Cluster'] == cluster) &
                              (cluster_mfi['donor_ID'] == donor) &
                              (cluster_mfi['drug_condition'] == 'CHX')]['FJComp-Alexa Fluor 488-A'].mean()

        mfi_oligo = cluster_mfi[(cluster_mfi['Cluster'] == cluster) &
                                (cluster_mfi['donor_ID'] == donor) &
                                (cluster_mfi['drug_condition'] == 'OLIGO')]['FJComp-Alexa Fluor 488-A'].mean()

        # Step 7: Calculate the two parameters: 'protein synthesis' and 'mitochondrial dependence'

        # Protein Synthesis: (MFI DMSO) - (MFI CHX)
        protein_synthesis = mfi_dms0 - mfi_chx

        # Mitochondrial Dependence:
        # [((MFI DMSO) - (MFI OLIGO)) / ((MFI DMSO) - (MFI CHX))] * 100
        if (mfi_dms0 - mfi_chx) != 0:  # Avoid division by zero
            mitochondrial_dependence = ((mfi_dms0 - mfi_oligo) / (mfi_dms0 - mfi_chx)) * 100
        else:
            mitochondrial_dependence = None  # Handle division by zero

        # Step 8: Append the results for the current cluster and donor to the results list
        results.append({
            'Cluster': cluster,
            'donor_ID': donor,
            'protein_synthesis': protein_synthesis,
            'mitochondrial_dependence': mitochondrial_dependence
        })

# Step 9: Convert the results list into a DataFrame
df_results = pd.DataFrame(results)
print(df_results)

# VISUALIZE mitochondrial dependencies across clusters using a color-coded UMAP

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# PRISM-LIKE STYLE
# ============================================================

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 5
mpl.rcParams["ytick.labelsize"] = 5
mpl.rcParams["legend.fontsize"] = 5
mpl.rcParams["axes.linewidth"] = 0.6
mpl.rcParams["xtick.major.width"] = 0.6
mpl.rcParams["ytick.major.width"] = 0.6
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

sns.set_style("white")

# ============================================================
# CREATE UMAP DATAFRAME
# ============================================================

df_umap = pd.DataFrame(
    umap_result,
    columns=["UMAP1", "UMAP2"]
)

df_umap["Cluster"] = clusters

df_umap = df_umap.merge(
    df_results[["Cluster", "mitochondrial_dependence", "donor_ID"]],
    on="Cluster",
    how="left"
)

# ============================================================
# SCALE MITOCHONDRIAL DEPENDENCE TO 0-100
# ============================================================

scaler = MinMaxScaler(feature_range=(0, 100))

df_umap["mitochondrial_dependence_scaled"] = scaler.fit_transform(
    df_umap[["mitochondrial_dependence"]]
)

vmin = 0
vmax = 100

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(2.75, 2))

scatter = ax.scatter(
    df_umap["UMAP1"],
    df_umap["UMAP2"],
    c=df_umap["mitochondrial_dependence_scaled"],
    cmap="YlGnBu",
    s=0.15,
    alpha=0.8,
    linewidths=0,
    vmin=vmin,
    vmax=vmax,
    rasterized=True
)

# ============================================================
# COLORBAR
# ============================================================

cbar = fig.colorbar(
    scatter,
    ax=ax,
    fraction=0.046,
    pad=0.03
)

cbar.set_label(
    "Mitochondrial dependence (%)",
    rotation=270,
    labelpad=8,
    fontsize=6
)

cbar.ax.tick_params(
    labelsize=5,
    width=0.6,
    length=2
)

cbar.outline.set_linewidth(0.6)

# ============================================================
# AXES
# ============================================================

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("")

ax.tick_params(
    axis="both",
    which="major",
    labelsize=5,
    width=0.6,
    length=2
)

sns.despine(ax=ax)

# ============================================================
# SAVE
# ============================================================

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "UMAP_clusters_mitochondrial_dependence.pdf"
    ),
    dpi=1200,
    bbox_inches="tight",
    format="pdf"
)

plt.show()

## Correlation in mitochondrial dependence between donors within each cluster

In [ ]:
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# ============================================================
# PRISM-LIKE STYLE
# ============================================================

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 5
mpl.rcParams["ytick.labelsize"] = 5
mpl.rcParams["legend.fontsize"] = 5
mpl.rcParams["axes.linewidth"] = 0.6
mpl.rcParams["xtick.major.width"] = 0.6
mpl.rcParams["ytick.major.width"] = 0.6
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

sns.set_style("white")

# ============================================================
# DONOR-DONOR PIVOT
# ============================================================

df_pivot = (
    df_results[df_results["donor_ID"].isin(["donor24M", "donor31F"])]
    .pivot(
        index="Cluster",
        columns="donor_ID",
        values="mitochondrial_dependence"
    )
    .reset_index()
)

df_pivot = df_pivot.dropna(subset=["donor24M", "donor31F"])

df_pivot["fold_difference"] = df_pivot["donor24M"] / df_pivot["donor31F"]

# ============================================================
# CORRELATION
# ============================================================

correlation, p_value = pearsonr(
    df_pivot["donor24M"],
    df_pivot["donor31F"]
)

if p_value < 0.0001:
    p_text = "****"
elif p_value < 0.001:
    p_text = "***"
elif p_value < 0.01:
    p_text = "**"
elif p_value < 0.05:
    p_text = "*"
else:
    p_text = "ns"

correlation_text = f"r = {correlation:.2f}\n{p_text}"

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(2.5, 2))

sns.scatterplot(
    data=df_pivot,
    x="donor24M",
    y="donor31F",
    hue="Cluster",
    palette="tab10",
    s=10,
    edgecolor="black",
    linewidth=0.25,
    marker="o",
    ax=ax,
    legend=False
)

sns.regplot(
    data=df_pivot,
    x="donor24M",
    y="donor31F",
    scatter=False,
    color="black",
    line_kws={
        "linewidth": 0.6,
        "ls": "--"
    },
    ax=ax
)

# ============================================================
# CORRELATION TEXT
# ============================================================

ax.text(
    0.05,
    0.95,
    correlation_text,
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=8
)

# ============================================================
# AXES
# ============================================================

ax.set_xlabel(
    "Mitochondrial dependence (%)\nDonor 1"
)

ax.set_ylabel(
    "Mitochondrial dependence (%)\nDonor 2"
)

ax.set_title("")

ax.tick_params(
    axis="both",
    which="major",
    labelsize=5,
    width=0.6,
    length=2,
    pad=1
)

ax.grid(False)
sns.despine(ax=ax)

# ============================================================
# SAVE PDF ONLY
# ============================================================

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Correlation_mitochondrial_dependence_across_2_donors_1p5x1p5.pdf"
    ),
    dpi=300,
    bbox_inches="tight",
    format="pdf"
)

plt.show()

print(f"Pearson Correlation: {correlation:.3f}")
print(f"P-value: {p_value:.3e}")
print(f"Significance: {p_text}")

# VISUALIZE PROTEIN SYNTHESIS

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# PRISM-LIKE STYLE
# ============================================================

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 5
mpl.rcParams["ytick.labelsize"] = 5
mpl.rcParams["legend.fontsize"] = 5
mpl.rcParams["axes.linewidth"] = 0.6
mpl.rcParams["xtick.major.width"] = 0.6
mpl.rcParams["ytick.major.width"] = 0.6
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

sns.set_style("white")

# ============================================================
# CREATE UMAP DATAFRAME
# ============================================================

df_umap = pd.DataFrame(
    umap_result,
    columns=["UMAP1", "UMAP2"]
)

df_umap["Cluster"] = clusters

df_umap = df_umap.merge(
    df_results[["Cluster", "protein_synthesis"]],
    on="Cluster",
    how="left"
)

# ============================================================
# SCALE PROTEIN SYNTHESIS TO 0-100
# ============================================================

scaler = MinMaxScaler(feature_range=(0, 100))

df_umap["protein_synthesis_scaled"] = scaler.fit_transform(
    df_umap[["protein_synthesis"]]
)

vmin = 0
vmax = 100

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(2.75, 2))

scatter = ax.scatter(
    df_umap["UMAP1"],
    df_umap["UMAP2"],
    c=df_umap["protein_synthesis_scaled"],
    cmap="Reds",
    s=0.15,
    alpha=0.8,
    linewidths=0,
    vmin=vmin,
    vmax=vmax,
    rasterized=True
)

# ============================================================
# COLORBAR
# ============================================================

cbar = fig.colorbar(
    scatter,
    ax=ax,
    fraction=0.046,
    pad=0.03
)

cbar.set_label(
    "Protein synthesis (%)",
    rotation=270,
    labelpad=8,
    fontsize=6
)

cbar.ax.tick_params(
    labelsize=5,
    width=0.6,
    length=2
)

cbar.outline.set_linewidth(0.6)

# ============================================================
# AXES
# ============================================================

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("")

ax.tick_params(
    axis="both",
    which="major",
    labelsize=5,
    width=0.6,
    length=2
)

sns.despine(ax=ax)

# ============================================================
# SAVE
# ============================================================

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "UMAP_clusters_protein_synthesis.pdf"
    ),
    dpi=1200,
    bbox_inches="tight",
    format="pdf"
)

plt.show()

## Correlation in protein synthesis between donors within each cluster

In [ ]:
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# ============================================================
# PRISM-LIKE STYLE
# ============================================================

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 5
mpl.rcParams["ytick.labelsize"] = 5
mpl.rcParams["legend.fontsize"] = 5
mpl.rcParams["axes.linewidth"] = 0.6
mpl.rcParams["xtick.major.width"] = 0.6
mpl.rcParams["ytick.major.width"] = 0.6
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

sns.set_style("white")

# ============================================================
# DONOR-DONOR PIVOT
# ============================================================

df_pivot = (
    df_results[df_results["donor_ID"].isin(["donor24M", "donor31F"])]
    .pivot(
        index="Cluster",
        columns="donor_ID",
        values="protein_synthesis"
    )
    .reset_index()
)

df_pivot = df_pivot.dropna(
    subset=["donor24M", "donor31F"]
)

df_pivot["fold_difference"] = (
    df_pivot["donor24M"] /
    df_pivot["donor31F"]
)

# ============================================================
# CORRELATION
# ============================================================

correlation, p_value = pearsonr(
    df_pivot["donor24M"],
    df_pivot["donor31F"]
)

if p_value < 0.0001:
    p_text = "****"
elif p_value < 0.001:
    p_text = "***"
elif p_value < 0.01:
    p_text = "**"
elif p_value < 0.05:
    p_text = "*"
else:
    p_text = "ns"

correlation_text = f"r = {correlation:.2f}\n{p_text}"

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(2.5, 2))

sns.scatterplot(
    data=df_pivot,
    x="donor24M",
    y="donor31F",
    hue="Cluster",
    palette="tab10",
    s=10,
    edgecolor="black",
    linewidth=0.25,
    marker="o",
    ax=ax,
    legend=False
)

sns.regplot(
    data=df_pivot,
    x="donor24M",
    y="donor31F",
    scatter=False,
    color="black",
    line_kws={
        "linewidth": 0.6,
        "ls": "--"
    },
    ax=ax
)

# ============================================================
# CORRELATION TEXT
# ============================================================

ax.text(
    0.05,
    0.95,
    correlation_text,
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=8
)

# ============================================================
# AXES
# ============================================================

ax.set_xlabel(
    "Protein synthesis \nDonor 1"
)

ax.set_ylabel(
    "Protein synthesis \nDonor 2"
)

ax.set_title("")

ax.tick_params(
    axis="both",
    which="major",
    labelsize=5,
    width=0.6,
    length=2,
    pad=1
)

ax.grid(False)
sns.despine(ax=ax)

# ============================================================
# SAVE
# ============================================================

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Correlation_protein_synthesis_across_2_donors_1p5x1p5.pdf"
    ),
    dpi=300,
    bbox_inches="tight",
    format="pdf"
)

plt.show()

print(f"Pearson Correlation: {correlation:.3f}")
print(f"P-value: {p_value:.3e}")
print(f"Significance: {p_text}")